# Day 52: XGBoost & LightGBM

## Install Note
`GradientBoostingClassifier` from Day 51 is scikit-learn's built-in gradient boosting implementation, but in practice most production and competition-winning models use XGBoost or LightGBM instead - both implement the same core gradient boosting idea but with major engineering improvements: faster training (parallelized tree construction, histogram-based splitting), built-in regularization to fight overfitting, native handling of missing values, and better scalability to large datasets. Because they're separate, actively-maintained libraries rather than part of scikit-learn, they need to be installed separately before use. Both still follow scikit-learn's familiar `.fit()`/`.predict()`/`.score()` interface, so switching to them from `GradientBoostingClassifier` requires very little code change.

In [ ]:
# run this once in your terminal (or a cell with a leading !) before importing xgboost/lightgbm below
# pip install xgboost lightgbm

## XGBoost Classifier
XGBoost (Extreme Gradient Boosting) builds on the same sequential, residual-correcting idea as gradient boosting but adds explicit regularization terms (penalizing tree complexity directly in its objective function, similar in spirit to Ridge/Lasso penalizing large coefficients), efficient handling of sparse/missing data, and a highly optimized, parallelized training routine - which is why it has been a dominant choice in Kaggle competitions and industry for structured/tabular data for years. `n_estimators=100` and `learning_rate=0.1` control the same tradeoff as in plain gradient boosting: more, smaller sequential correction steps. `eval_metric='mlogloss'` specifies multi-class log loss as the metric XGBoost tracks internally during training (appropriate here since Iris has 3 classes) - it's about how confidently correct the predicted probabilities are, not just whether the final predicted label is right. The printed accuracy lets you compare directly against the scikit-learn `GradientBoostingClassifier` result from Day 51 on the same Iris data.

In [ ]:
from xgboost import XGBClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# n_estimators/learning_rate control boosting rounds and step size, same idea as sklearn's GradientBoostingClassifier;
# eval_metric='mlogloss' sets the internal loss XGBoost optimizes/tracks for this multi-class problem
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, eval_metric='mlogloss', random_state=42)
xgb_model.fit(X_train, y_train)
print('XGBoost accuracy:', xgb_model.score(X_test, y_test))

## LightGBM Classifier
LightGBM is another gradient boosting library, developed by Microsoft, optimized specifically for speed and memory efficiency on large datasets. The key architectural difference from XGBoost's default behavior is how trees grow: most gradient boosting implementations grow trees "level-wise" (expanding every node at the current depth before going deeper), while LightGBM grows trees "leaf-wise" - it always splits whichever single leaf will reduce the loss the most, regardless of depth. This tends to produce more accurate trees for a given number of leaves and trains faster, though leaf-wise growth can overfit more easily on small datasets (like this small Iris example) since it can produce deeper, more complex trees than level-wise growth would for the same leaf count - worth keeping in mind if this model overfits noticeably compared to XGBoost. On large, real-world datasets (hundreds of thousands+ rows), LightGBM's speed advantage becomes much more noticeable than it will here.

In [ ]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(n_estimators=100, random_state=42)  # leaf-wise tree growth, fast on large datasets
lgbm_model.fit(X_train, y_train)
print('LightGBM accuracy:', lgbm_model.score(X_test, y_test))

## Feature Importance
One practical advantage of tree-based ensembles like XGBoost/LightGBM (shared with Random Forest) is that they can report feature importance almost for free, since it's a natural byproduct of how trees are built - both libraries track, across all the boosted trees, how often (or how much) each feature was used to split the data and how much those splits reduced the loss. A feature that gets used in many important splits across many trees is considered more "important" to the model's predictions. This is genuinely useful for real projects: it helps explain model behavior to stakeholders, guides feature selection (dropping consistently low-importance features), and can surface data quality issues (an unexpectedly dominant feature might indicate a data leak). The code below pulls `feature_importances_` from the trained XGBoost model and sorts the 4 Iris features from most to least important - for Iris, expect petal length/width to dominate over sepal measurements, since petal dimensions separate the species most cleanly.

In [ ]:
import pandas as pd

feature_names = load_iris().feature_names
importances = pd.Series(xgb_model.feature_importances_, index=feature_names)  # relative importance score per feature, from the trained XGBoost model
print(importances.sort_values(ascending=False))  # highest importance first